In [2]:
import csv
import pandas as pd
from pathlib import Path
# --------------------------
# 0) Chemins du projet
# --------------------------
CODE_DIR = Path().resolve()              # .../SYNCOGEST/Code/Codes VICON
PROJECT_ROOT = CODE_DIR.parent.parent    # .../SYNCOGEST
DATA_DIR = PROJECT_ROOT / "DATA"

VICON_DIR = DATA_DIR / "VICON_CSV"
EXCEL_DIR = DATA_DIR / "Excels_code"

print("PROJECT_ROOT =", PROJECT_ROOT)
print("VICON_DIR =", VICON_DIR)
print("EXCEL_DIR =", EXCEL_DIR)

def read_vicon_csv(csv_path):
    """
    Parse un CSV Vicon avec:
    L1: Trajectories
    L2: 100
    L3: noms marqueurs (avec vides)
    L4: Frame/Sub Frame puis X/Y/Z
    L5: unités (mm)
    L6+: données
    """
    with open(csv_path, "r", encoding="utf-8", errors="replace", newline="") as f:
        reader = csv.reader(f)
        header_lines = [next(reader) for _ in range(5)]

    marker_row = header_lines[2]   # ligne 3
    axis_row   = header_lines[3]   # ligne 4

    n = max(len(marker_row), len(axis_row))
    marker_row += [""] * (n - len(marker_row))
    axis_row   += [""] * (n - len(axis_row))

    # forward-fill des marqueurs (car Y/Z sont vides)
    filled = []
    last = ""
    for m in marker_row:
        m = (m or "").strip()
        if m == "":
            filled.append(last)
        else:
            last = m
            filled.append(last)

    # construit des noms plats: "<marker>_X", "<marker>_Y", "<marker>_Z"
    colnames = []
    for m, a in zip(filled, axis_row):
        m = (m or "").strip()
        a = (a or "").strip()

        if a in ["Frame", "Sub Frame"]:
            colnames.append(a)
        elif a in ["X", "Y", "Z"]:
            colnames.append(f"{m}_{a}")
        else:
            colnames.append(m if m else a)

    df = pd.read_csv(csv_path, skiprows=5, header=None, names=colnames, engine="python")
    df = df.dropna(axis=1, how="all")  # supprime colonnes vides (trailing commas)
    return df

PROJECT_ROOT = /Users/matysprecloux/Desktop/SYNCOGEST
VICON_DIR = /Users/matysprecloux/Desktop/SYNCOGEST/DATA/VICON_CSV
EXCEL_DIR = /Users/matysprecloux/Desktop/SYNCOGEST/DATA/Excels_code


In [3]:
import re
import numpy as np

def find_xyz_cols(cols, token):
    """
    token: 'epaule_D', 'epaule_G', '2epaule_D', '2epaule_G'
    Match: n'importe quel préfixe avant ':', ex 'Patient 1:epaule_D_X'
    """
    pat = re.compile(rf"(?:^|:)\s*{re.escape(token)}_([XYZ])\b", re.IGNORECASE)
    found = {}
    for c in cols:
        c2 = c.replace(" ", "")  # au cas où
        m = pat.search(c2)
        if m:
            axis = m.group(1).upper()
            if axis not in found or len(c) < len(found[axis]):
                found[axis] = c
    return found.get("X"), found.get("Y"), found.get("Z")

In [4]:
def shoulder_width_vicon(csv_path):
    df = read_vicon_csv(csv_path)
    cols = list(df.columns)

    out = {"csv": str(csv_path)}

    for pid, prefix in [("P1", ""), ("P2", "2")]:
        tokenD = f"{prefix}epaule_D"
        tokenG = f"{prefix}epaule_G"

        RX, RY, RZ = find_xyz_cols(cols, tokenD)
        LX, LY, LZ = find_xyz_cols(cols, tokenG)

        if None in [RX, RY, RZ, LX, LY, LZ]:
            out[f"{pid}_error"] = f"missing {tokenD}/{tokenG} (X/Y/Z)"
            out[f"{pid}_found"] = str([RX,RY,RZ,LX,LY,LZ])
            continue

        sw = np.sqrt((df[RX]-df[LX])**2 + (df[RY]-df[LY])**2 + (df[RZ]-df[LZ])**2)
        sw = sw.replace([np.inf, -np.inf], np.nan).dropna()

        out[f"{pid}_shoulder_width_median_mm"] = float(sw.median())
        out[f"{pid}_shoulder_width_mean_mm"]   = float(sw.mean())
        out[f"{pid}_n_frames_used"]            = int(len(sw))

    return out

In [5]:
from pathlib import Path
import pandas as pd

ROOT = VICON_DIR
csv_files = list(ROOT.rglob("*.csv"))

rows = []
for f in csv_files:
    rows.append(shoulder_width_vicon(f))

df_out = pd.DataFrame(rows)

out_path = EXCEL_DIR / "vicon_shoulder_width_by_csv.xlsx"
df_out.to_excel(out_path, index=False)

print("✅ Saved:", out_path)

✅ Saved: /Users/matysprecloux/Desktop/SYNCOGEST/DATA/Excels_code/vicon_shoulder_width_by_csv.xlsx
